In [0]:
import pandas as pd
import numpy as np
import requests
import plotly.express as px
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import plotly.graph_objects as go
from scipy.stats import norm



# #1 DATA ACQUISIITON

### Corn yields from Quickstat

In [0]:
API_KEY = "0DC0FD85-4E88-3CDE-A23D-7B107154428D"
url = "http://quickstats.nass.usda.gov/api/api_GET/"

In [0]:
def fetch_yield(agg_level, state=None, county=None):
    params = {
        "key": API_KEY,
        "commodity_desc": "CORN",
        "statisticcat_desc": "YIELD",
        "unit_desc": "BU / ACRE",
        "agg_level_desc": agg_level,
        "freq_desc": "ANNUAL",
        "format": "JSON"
    }
    if state:
        params["state_name"] = state
    if county:
        params["county_name"] = county

    resp = requests.get(url, params=params)

    
    data = resp.json().get("data", [])


    if not data:
        print(f"No data returned for {state}, {county}")
        return pd.DataFrame()

    df = pd.DataFrame(data)

    if "Value" not in df.columns:
        print(f"'Value' column missing for {state}, {county}")
        return pd.DataFrame()

    df = df[df["Value"].str.contains(r"^\d", na=False)]
    df["yield_bu_per_acre"] = df["Value"].str.replace(",", "").astype(float)
    df["year"] = df["year"].astype(int)
    df["level"] = agg_level.lower()

    return df


yield_national = fetch_yield("NATIONAL")
yield_state = fetch_yield("STATE")


yield_national=yield_national[yield_national['short_desc']=='CORN, GRAIN - YIELD, MEASURED IN BU / ACRE']
yield_national=yield_national[yield_national['reference_period_desc']=='YEAR']

yield_national=yield_national[['year','yield_bu_per_acre']]
yield_national.reset_index(inplace=True,drop=True)

yield_state=yield_state[yield_state['short_desc']=='CORN, GRAIN - YIELD, MEASURED IN BU / ACRE']
yield_state=yield_state[yield_state['reference_period_desc']=='YEAR']


yield_state=yield_state[['year','yield_bu_per_acre','state_name','state_ansi']]
yield_state.reset_index(inplace=True,drop=True)




In [0]:
STATE_LIST=['ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY','MICHIGAN',
       'MINNESOTA', 'MISSOURI', 'NEBRASKA', 'OHIO', 'SOUTH DAKOTA',
       'TENNESSEE','WISCONSIN']


all_counties_df = pd.DataFrame()

for i in STATE_LIST:
    print(f"Fetching {i}...")
    counties = fetch_yield("COUNTY",state=i)
    all_counties_df = pd.concat([all_counties_df, counties], ignore_index=True)
    

In [0]:
all_counties_df=all_counties_df[all_counties_df['short_desc']=='CORN, GRAIN - YIELD, MEASURED IN BU / ACRE']
all_counties_df=all_counties_df[['year','yield_bu_per_acre','state_name','state_ansi','county_name','county_code']]
yield_county=all_counties_df.copy()

### Weather data

In [0]:
weather = pd.read_parquet("./hist_wx_df.parquet", engine="pyarrow") 

weather.rename(columns={"adm2_name": "county_name"}, inplace=True)
weather.rename(columns={"adm1_name": "state_name"}, inplace=True)

weather['date'] = pd.to_datetime(weather['date'])
weather['year']= weather['date'].dt.year
weather = weather[weather["date"].dt.month.between(5, 9)]

#ASSIGNING SEASON TO EACH ROW
weather["season"] = weather["date"].apply(lambda d: f"{d.year}/{d.year + 1}" if d.month >= 9 else f"{d.year - 1}/{d.year}")

# #2 DATA SANITY CHECK

In [0]:
weather_cols = ["tmax", "tmin", "precip", "swvl1", "swvl2"]
na_counts = weather[weather_cols].isna().sum()
na_counts



No missing values in the df

Checking for consistency below

In [0]:
# Check date type and range
print(weather['date'].dtype)
print(f"Date range: {weather['date'].min()} to {weather['date'].max()}")

inconsistent_temp = weather[weather['tmax'] < weather['tmin']]
print(f"Number of rows where tmax < tmin: {len(inconsistent_temp)}")


negative_precip = weather[weather['precip'] < 0]
print(f"Number of rows with negative precipitation: {len(negative_precip)}")

# Example: check for out-of-range values
out_of_range = weather[
    (weather['tmax'] < -50) | (weather['tmax'] > 60) |
    (weather['tmin'] < -50) | (weather['tmin'] > 60) |
    (weather['precip'] < 0) | (weather['precip'] > 500) |
    (weather['swvl1'] < 0) | (weather['swvl1'] > 1) |
    (weather['swvl2'] < 0) | (weather['swvl2'] > 1)
]
print(f"Rows with out-of-range values: {len(out_of_range)}")


# #3 EXPLORATORY DATA ANALYSIS

### YIELD TRENDS

In [0]:
fig = px.line(
    yield_national.sort_values("year"),
    x="year",
    y="yield_bu_per_acre",
    title="U.S. National Corn Yield Over Time",
    markers=True,
    labels={"yield_bu_per_acre": "Bushels per Acre"}
)
fig.show()


US national yield increased a lot over the last 50y and now may start platoing. 2012 drought can be easily pointed out

In [0]:
fig = px.line(
    yield_state.sort_values("year"),
    x="year",
    y="yield_bu_per_acre",
    color="state_name",
    title="Corn Yield Over Time by State",
    markers=True,
    labels={"yield_bu_per_acre": "Bushels per Acre"},
)

fig.show()


Big disparities between the states, but the overall trend is an increase from the 50's to 2020's

In [0]:
df=yield_county.copy()
TOP_5_CORN_STATES = [
    "IOWA",
    "ILLINOIS",
    "NEBRASKA",
    "MINNESOTA",
    "INDIANA"
]

for state in TOP_5_CORN_STATES:
    df_state = df[df["state_name"] == state].sort_values("year")

    fig = px.line(
        df_state,
        x="year",
        y="yield_bu_per_acre",
        color="county_name",
        title=f"Yield Trends by County in {state}",
        labels={"yield_bu_per_acre": "Bushels per Acre", "year": "Year"},
    )

    fig.show()


# #4 Feature Engineering

In the weather dataframe we have got continues time series. Our goal is divide the year in a few stages important to corn crop development and to go from a continuous dataset to a discrete one. Hence we are going to build new features.

We are going to divide the crop year into 3 different parts, each one being representative of a few stages of corn crop development:
- Early seaason from May to June
- Pollination period: July
- Grain fill from August to September

We will consider the Corn belt representative of the entirety of the US, hence harvest should be beginning late September and planting happens in May. Therefore I will not be using all of the weather data between November and April.


We add GDD: grwing Degree Days features as it reflects heat accumulation critical for crop growth.As well as stress indicators, reflecting the number of days above 35 degrees or below 0, as prolonged exposure to such temperatures can reduce yield

In [0]:

def assign_stage(row):
    month = row["date"].month
    if month in [5, 6]:
        return "early"
    elif month == 7:
        return "mid"
    elif month in [8, 9]:
        return "late"
    else:
        return None
    
weather['GDD'] = ((weather['tmax'] + weather['tmin']) / 2 - 10).clip(lower=0)

weather["heat_stress"] = (weather["tmax"] > 35).astype(int)
weather["frost"] = (weather["tmin"] < 0).astype(int)

weather["stage"] = weather.apply(assign_stage, axis=1)
weather = weather[weather["stage"].notna()]


agg_weather = weather.groupby(["year", "state_name", "county_name", "stage"]).agg({
    "tmax": "mean",
    "tmin": "mean",
    "precip": "sum",
    "swvl1": "mean",
    "swvl2": "mean",
    "GDD":"mean",
    "heat_stress": "sum",
    "frost": "sum"
}).reset_index()

final_weather = agg_weather.pivot(index=["year", "state_name", "county_name"], 
                        columns="stage", 
                        values=["tmax", "tmin", "precip", "swvl1", "swvl2","GDD","heat_stress","frost"])

# Flatten the column names
final_weather.columns = ['_'.join(col).strip() for col in final_weather.columns.values]
final_weather = final_weather.reset_index()
final_weather = final_weather.sort_values(by="state_name", ascending=True).reset_index(drop=True)




In [0]:
final_weather["state_name"] = final_weather["state_name"].str.upper()
final_weather["county_name"] = final_weather["county_name"].str.upper()

state_with_data=final_weather.state_name.unique().tolist()

yield_county = yield_county[yield_county["state_name"].isin(state_with_data)]
yield_county=yield_county[['year','state_name','county_name', 'yield_bu_per_acre']]
weather_yield=pd.merge(final_weather,yield_county,on=["year", "state_name", "county_name"],how="inner")

Firstly I will begin to work on a State's level. Therefore, I will take the average values of the weather data across all counties to get a number for each State. 

In [0]:
# Getting the average values by year and state, we could have considered each county harvested area and then do a weighted average to get closer to reality figures
data_avg_by_state = weather_yield.groupby(['year', 'state_name'], as_index=False).mean(numeric_only=True)

#merging with yield reported by state on quickstat to see if taking the average yield is a source of big differencies 
yield_state.rename(columns={'yield_bu_per_acre': 'yield_state'}, inplace=True)
yield_state=yield_state[['year', 'state_name', 'yield_state']]

data_avg_by_state=pd.merge(yield_state, data_avg_by_state, on=["year", "state_name"], how="inner")


In [0]:

weather_cols = ['tmin_early','tmin_late','tmin_mid','tmax_early','tmax_late','tmax_mid',
    'precip_early', 'precip_mid', 'precip_late',
    'swvl1_early', 'swvl1_mid', 'swvl1_late',
    'swvl2_early', 'swvl2_mid', 'swvl2_late',
    'GDD_early', 'GDD_mid', 'GDD_late',
    'heat_stress_early', 'heat_stress_mid', 'heat_stress_late',
    'frost_early', 'frost_mid', 'frost_late']

Below I am cheking if the correlations between the official state yield and the yield calculated by making an average of all counties' yield (and not a weighted average using total surface harvested) are the same. 

In [0]:
# Correlation by state with mean yield
corr_by_state = data_avg_by_state.groupby('state_name')[weather_cols + ['yield_bu_per_acre']].corr().unstack()['yield_bu_per_acre']
corr_by_state


In [0]:
# Correlation by state with official reported yield
corr_by_state_off = data_avg_by_state.groupby('state_name')[weather_cols + ['yield_state']].corr().unstack()['yield_state']
corr_by_state_off


In [0]:
'''
state = "IOWA"  # change to any state

data_avg_by_state_state = data_avg_by_state[data_avg_by_state['state_name'] == state]

for col in weather_cols:
    fig = px.scatter(data_avg_by_state_state, x=col, y='yield_bu_per_acre', trendline="ols",
                     title=f"Yield vs {col} in {state}")
    fig.show()'''



In [0]:
corr_table = []

for state, group in data_avg_by_state.groupby('state_name'):
    corrs = group[weather_cols + ['yield_bu_per_acre']].corr()['yield_bu_per_acre'].drop('yield_bu_per_acre')
    corrs.name = state
    corr_table.append(corrs)

corr_df = pd.DataFrame(corr_table)
corr_df


plt.figure(figsize=(14, 8))
sns.heatmap(corr_df, cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.5)

plt.title("Correlation between Weather Variables and Yield by State")
plt.xlabel("Weather Variable")
plt.ylabel("State")
plt.tight_layout()
plt.show()



In [0]:
corr_table = []

for state, group in data_avg_by_state.groupby('state_name'):
    corrs = group[weather_cols + ['yield_state']].corr()['yield_state'].drop('yield_state')
    corrs.name = state
    corr_table.append(corrs)

corr_df = pd.DataFrame(corr_table)
corr_df


plt.figure(figsize=(14, 8))
sns.heatmap(corr_df, cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.5)

plt.title("Correlation between Weather Variables and Yield by State")
plt.xlabel("Weather Variable")
plt.ylabel("State")
plt.tight_layout()
plt.show()



Overall it seems that the most significant waether variables are always the one of the mid-season window

In [0]:
data_avg_by_state[weather_cols].corr()

To reduce multicollinearity and dimension, we are only going to use GDD variables to account for temperature variations, as it accounts for Tmin and Tmax. Moreover, the soil moisture levels swvl1 and swvl2 are also  almost perfectly correlated no matter the stage, hence we are only keeping swvl1 variables which have in average a higher correlation with the yield

In [0]:
data_avg_by_state=data_avg_by_state[['state_name','year', 'precip_early', 'precip_late',
       'precip_mid', 'swvl1_early', 'swvl1_late', 'swvl1_mid', 'GDD_early', 'GDD_late', 'GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid','yield_bu_per_acre','yield_state']]

weather_cols=['precip_early', 'precip_late',
       'precip_mid', 'swvl1_early', 'swvl1_late', 'swvl1_mid', 'GDD_early', 'GDD_late', 'GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid','yield_bu_per_acre','yield_state']



I am going to use StandardScaller scalling to make all features fit stricly in the same range, which is useful for some kind of models

In [0]:
cols_to_scale = ['precip_early', 'precip_late', 'precip_mid', 'GDD_early', 'GDD_late','GDD_mid','swvl1_early','swvl1_late','swvl1_mid']

scaler = StandardScaler()

data_avg_by_state_scaled=data_avg_by_state.copy()
data_avg_by_state_scaled[cols_to_scale] = scaler.fit_transform(data_avg_by_state[cols_to_scale])
data_avg_by_state_scaled

# #5 Model Development & Evaluation

In [0]:
states=data_avg_by_state['state_name'].unique().tolist()
feature_cols=['precip_early', 'precip_late', 'precip_mid',
       'swvl1_early', 'swvl1_late', 'swvl1_mid', 'GDD_early', 'GDD_late',
       'GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid']


In [0]:
models_scale_needed = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "SVR": SVR()
}

results_scaled = {}

for state in states:
    df_state = data_avg_by_state_scaled[data_avg_by_state_scaled['state_name'] == state].dropna(subset=['yield_state'])
    X = df_state[feature_cols].values
    y = df_state['yield_state'].values

    state_results = {}
    for name, model in models_scale_needed.items():
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mse_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
        r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
        state_results[name] = {
            'mse_mean': np.mean(mse_scores),
            'r2_mean': np.mean(r2_scores)
        }
    results_scaled[state] = state_results

results_df_scaled = pd.DataFrame({state: {model: vals['r2_mean'] for model, vals in models_dict.items()} 
                                for state, models_dict in results_scaled.items()}).T
results_df_scaled

In [0]:
models_no_scale = {
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=50),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results_scaled = {}

for state in states:
    df_state = data_avg_by_state[data_avg_by_state['state_name'] == state].dropna(subset=['yield_state'])
    X = df_state[feature_cols].values
    y = df_state['yield_state'].values

    state_results = {}
    for name, model in models_no_scale.items():
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mse_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
        r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
        state_results[name] = {
            'mse_mean': np.mean(mse_scores),
            'r2_mean': np.mean(r2_scores)
        }
    results_scaled[state] = state_results

# Convert results to DataFrame for easier comparison
results_df_scaled = pd.DataFrame({state: {model: vals['r2_mean'] for model, vals in models_dict.items()} 
                                for state, models_dict in results_scaled.items()}).T



### As the models performance are poor state's level, let's work bottom up from the counties scale first

In [0]:
weather_yield=weather_yield[['state_name','county_name','year', 'precip_early', 'precip_late',
       'precip_mid', 'swvl1_early', 'swvl1_late', 'swvl1_mid', 'GDD_early', 'GDD_late', 'GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid',
       'yield_bu_per_acre']]

cols_to_scale = ['precip_early', 'precip_late', 'precip_mid', 'GDD_early', 'GDD_late','GDD_mid']

scaler = StandardScaler()

weather_yield_scaled=weather_yield.copy()
weather_yield_scaled[cols_to_scale] = scaler.fit_transform(weather_yield_scaled[cols_to_scale])
weather_yield_scaled

In [0]:
feature_cols = ['precip_early', 'precip_late', 'precip_mid', 'GDD_early', 'GDD_late','GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid']

target_col = 'yield_bu_per_acre'


results_by_county = {}

for (state, county), df_group in weather_yield_scaled.groupby(['state_name', 'county_name']):
    df_group = df_group.dropna(subset=[target_col])
    if len(df_group) < 5:
        continue 
    
    X = df_group[feature_cols].values
    y = df_group[target_col].values

    model_scores = {}
    for name, model in models_scale_needed.items():
        kf = KFold(n_splits=2, shuffle=True, random_state=42)
        mse_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
        r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
        model_scores[name] = {
            'mse_mean': np.mean(mse_scores),
            'r2_mean': np.mean(r2_scores)
        }
    
    results_by_county[(state, county)] = model_scores


results_df_by_county_scaled = pd.DataFrame({
    (state, county): {model: scores['r2_mean'] for model, scores in model_dict.items()}
    for (state, county), model_dict in results_by_county.items()
}).T


results_df_by_county_scaled.index.names = ['state', 'county']
results_df_by_county_scaled = results_df_by_county_scaled.reset_index()

In [0]:
average_r2_per_model = results_df_by_county_scaled.drop(columns=['state', 'county']).mean().sort_values(ascending=False)
print(average_r2_per_model)

best_models = results_df_by_county_scaled.drop(columns=['state', 'county']).idxmax(axis=1)

model_win_counts = best_models.value_counts()
print(model_win_counts)


average_r2_per_model.plot(kind='bar', title='Average R² per Model', ylabel='R²')
plt.tight_layout()
plt.show()

model_win_counts.plot(kind='bar', title='Number of Counties Where Model Performed Best', ylabel='Count')
plt.tight_layout()
plt.show()



In [0]:
feature_cols = ['precip_early', 'precip_late', 'precip_mid', 'GDD_early', 'GDD_late','GDD_mid','heat_stress_early', 'heat_stress_late', 'heat_stress_mid',
       'frost_early', 'frost_late', 'frost_mid']

target_col = 'yield_bu_per_acre'


results_by_county = {}

for (state, county), df_group in weather_yield.groupby(['state_name', 'county_name']):
    df_group = df_group.dropna(subset=[target_col])
    if len(df_group) < 5:
        continue 
    
    X = df_group[feature_cols].values
    y = df_group[target_col].values

    model_scores = {}
    for name, model in models_no_scale.items():
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mse_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
        r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
        model_scores[name] = {
            'mse_mean': np.mean(mse_scores),
            'r2_mean': np.mean(r2_scores)
        }
    
    results_by_county[(state, county)] = model_scores


results_df_by_county = pd.DataFrame({
    (state, county): {model: scores['r2_mean'] for model, scores in model_dict.items()}
    for (state, county), model_dict in results_by_county.items()
}).T


results_df_by_county.index.names = ['state', 'county']
results_df_by_county = results_df_by_county.reset_index()

In [0]:
average_r2_per_model = results_df_by_county.drop(columns=['state', 'county']).mean().sort_values(ascending=False)
print(average_r2_per_model)


best_models = results_df_by_county.drop(columns=['state', 'county']).idxmax(axis=1)

model_win_counts = best_models.value_counts()
print(model_win_counts)


average_r2_per_model.plot(kind='bar', title='Average R² per Model', ylabel='R²')
plt.tight_layout()
plt.show()

model_win_counts.plot(kind='bar', title='Number of Counties Where Model Performed Best', ylabel='Count')
plt.tight_layout()
plt.show()



Overall, all models are performing really badly, for the sake of the exercise I will chosing one among them and continue the project as it worked

# #6 Predict 2024 Yield


CREATING INPUT VARIABLES

2024 weather data can be created using historical averages over the last 5years

In [0]:
weather_2024 = (
    weather_yield
    .groupby(['state_name', 'county_name'])  
    .tail(5)  # last 5 years: 2019–2023
    .groupby(['state_name', 'county_name'])
    .mean()
    .reset_index()
)
weather_2024['year'] = 2024


Some columns need to be rounded to integers in order to stay consitent

In [0]:
cols_to_round=['heat_stress_early', 'heat_stress_late',
       'heat_stress_mid', 'frost_early', 'frost_late', 'frost_mid']
weather_2024[cols_to_round] = weather_2024[cols_to_round].apply(np.ceil)
weather_2024.drop(columns=['yield_bu_per_acre'], inplace=True)
weather_2024_yield=pd.merge(weather_2024,yield_county,on=["year", "state_name", "county_name"],how="inner")



As most of the models had close performances,

In [0]:
rf_model = models_no_scale['RandomForest']

predictions_2024 = []

for (state, county), df_group_train in weather_yield.groupby(['state_name', 'county_name']):
    
    df_group_2024 = weather_2024_yield[(weather_2024_yield['state_name'] == state) & (weather_2024_yield['county_name'] == county)]
    if df_group_2024.empty:
        continue
    
    df_group_train = df_group_train.dropna(subset=[target_col] + feature_cols)
    if len(df_group_train) < 5:
        continue

    X_train = df_group_train[feature_cols].values
    y_train = df_group_train[target_col].values
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predict for 2024
    X_2024 = df_group_2024[feature_cols].values
    y_true_2024 = df_group_2024[target_col].values

    y_pred_2024 = model.predict(X_2024)

    # Get uncertainty estimate (std dev of predictions from all trees)
    tree_preds = np.stack([tree.predict(X_2024) for tree in model.estimators_])
    y_std_2024 = np.std(tree_preds, axis=0)

    for i in range(len(y_pred_2024)):
        predictions_2024.append({
            'state': state,
            'county': county,
            'y_pred': y_pred_2024[i],
            'y_true': y_true_2024[i],
            'uncertainty': y_std_2024[i]
        })

predictions_df = pd.DataFrame(predictions_2024)


mse = mean_squared_error(predictions_df['y_true'], predictions_df['y_pred'])
r2 = r2_score(predictions_df['y_true'], predictions_df['y_pred'])
mae = mean_absolute_error(predictions_df['y_true'], predictions_df['y_pred'])

print(f"R² = {r2:.3f}, MSE = {mse:.3f}, MAE = {mae:.3f}")


In [0]:
predictions_df['uncertainty'].describe()


In [0]:


plt.figure(figsize=(10,6))
plt.errorbar(predictions_df['y_true'], predictions_df['y_pred'], 
             yerr=predictions_df['uncertainty'], fmt='o', alpha=0.6)
plt.plot([predictions_df['y_true'].min(), predictions_df['y_true'].max()],
         [predictions_df['y_true'].min(), predictions_df['y_true'].max()],
         'k--', label='Perfect prediction')
plt.xlabel("Observed Yield (bu/ac)")
plt.ylabel("Predicted Yield (bu/ac)")
plt.title("Predicted vs Observed Corn Yield (with Uncertainty)")
plt.legend()
plt.grid(True)
plt.show()


In [0]:
unique_states = predictions_df['state'].unique()

for state in unique_states:
    df_state = predictions_df[predictions_df['state'] == state].copy()
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_state['y_true'],
        y=df_state['y_pred'],
        error_y=dict(type='data', array=df_state['uncertainty'], visible=True),
        mode='markers',
        marker=dict(size=6, color='blue', opacity=0.6),
        text=df_state['county'],
        name='Counties'
    ))

    min_val = min(df_state['y_true'].min(), df_state['y_pred'].min())
    max_val = max(df_state['y_true'].max(), df_state['y_pred'].max())
    
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Perfect prediction'
    ))

    fig.update_layout(
        title=f"Predicted vs Observed Corn Yield with Uncertainty<br><sup>{state}</sup>",
        xaxis_title="Observed Yield (bu/ac)",
        yaxis_title="Predicted Yield (bu/ac)",
        width=800,
        height=600,
        legend=dict(x=0.02, y=0.98),
        template='plotly_white'
    )

    fig.show()


As we could have expected, our models are not performing well at predicting yield. Uncertainty remains high, and the models rarely predict accurate values. However, it's important to keep in mind that we did not have access to actual weather data for 2024 and the weather may not have been representative of the past five-year average.

The next step is to scale up from county-level yields to a national yield. To do this, I will extract harvested area data from Quick Stats, and compute the national yield as a weighted average of all counties' yields, using their respective harvested areas as weights.

In [0]:
def fetch_area(agg_level, state=None, county=None):
    params = {
        "key": API_KEY,
        "commodity_desc": "CORN",
        "statisticcat_desc": "AREA HARVESTED",
        "unit_desc": "ACRES",
        "agg_level_desc": agg_level,
        "freq_desc": "ANNUAL",
        "format": "JSON",
        "year__GE": 2024,
        "year__LE": 2024
    }
    if state:
        params["state_name"] = state
    if county:
        params["county_name"] = county

    resp = requests.get(url, params=params)

    
    data = resp.json().get("data", [])


    if not data:
        print(f"No data returned for {state}, {county}")
        return pd.DataFrame()

    df = pd.DataFrame(data)

    if "Value" not in df.columns:
        print(f"'Value' column missing for {state}, {county}")
        return pd.DataFrame()

    df = df[df["Value"].str.contains(r"^\d", na=False)]
    df["yield_bu_per_acre"] = df["Value"].str.replace(",", "").astype(float)
    df["year"] = df["year"].astype(int)
    df["level"] = agg_level.lower()

    return df





In [0]:
STATE_LIST=['ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY','MICHIGAN',
       'MINNESOTA', 'MISSOURI', 'NEBRASKA', 'OHIO', 'SOUTH DAKOTA',
       'TENNESSEE','WISCONSIN']


counties_area = pd.DataFrame()

for i in STATE_LIST:
    print(f"Fetching {i}...")
    counties = fetch_area("COUNTY",state=i)
    counties_area = pd.concat([counties_area, counties], ignore_index=True)
    

In [0]:
counties_area=counties_area[counties_area['short_desc']=='CORN, GRAIN - ACRES HARVESTED']
counties_area=counties_area[['state_name','county_name','Value']]
counties_area.rename(columns={'Value': 'area_harvested'}, inplace=True)

counties_area.rename(columns={'state_name': 'state','county_name': 'county'}, inplace=True)
predictions_df=predictions_df[['state', 'county', 'y_pred', 'y_true', 'uncertainty']]

Cleaning the counties names to ensure perfect match

In [0]:
def clean_string_column(df, column):
    return df[column].astype(str).str.strip().str.upper().str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


predictions_df['state_clean'] = clean_string_column(predictions_df, 'state')
predictions_df['county_clean'] = clean_string_column(predictions_df, 'county')

counties_area['state_clean'] = clean_string_column(counties_area, 'state')
counties_area['county_clean'] = clean_string_column(counties_area, 'county')

In [0]:
pred_w_area = pd.merge(predictions_df, counties_area, on=['state_clean', 'county_clean'], how='inner')
pred_w_area['area_harvested'] = pred_w_area['area_harvested'].str.replace(',', '', regex=False).astype(float)
pred_w_area['weight'] = pred_w_area['area_harvested'] / pred_w_area['area_harvested'].sum()
national_yield = (pred_w_area['y_pred'] * pred_w_area['weight']).sum()

In [0]:
# National yield
pred_w_area['weight'] = pred_w_area['area_harvested'] / pred_w_area['area_harvested'].sum()
national_yield = (pred_w_area['y_pred'] * pred_w_area['weight']).sum()

# National uncertainty
pred_w_area['weighted_variance'] = (pred_w_area['weight']**2) * (pred_w_area['uncertainty']**2)
national_std = (pred_w_area['weighted_variance'].sum())**0.5

print(f"National Yield: {national_yield:.2f} bu/acre")
print(f"Uncertainty (std dev): ±{national_std:.2f} bu/acre")

In [0]:
# 95% Confidence Interval
z = norm.ppf(0.975)
ci_low = national_yield - z * national_std
ci_high = national_yield + z * national_std

print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}] bu/acre")


# #7 Reporting

### MODELLING PROCESS


The first step of the modeling process was to create time windows to avoid working with continuous time series. I clearly defined three stages of the crop growing season, and all features were recalculated using either a sum or an average, depending on their nature.

Next, I examined multicollinearity between features and decided to remove the second soil moisture metric, as it was almost perfectly correlated with the first one.

After some research, I found that Growing Degree Days (GDD) is a commonly used metric to track crop development. It is calculated using minimum and maximum daily temperatures (Tmin and Tmax).

I also added stress indicators, defined as the number of days with extreme temperatures.

I tested six different types of models, ranging from linear regression to random forest. All of them performed poorly, both at the state and county levels.

For the sake of the exercise, I chose to select Random Forest models among them and continued the project as if the predictions were meaningful, even though the overall performance remained poor.

The models predicted yield on a county-by-county basis, which I then extrapolated to produce a national yield estimate by weighting each county's yield by its corresponding harvested area.

The final prediction was 166.54 bu/acre, with a standard deviation of 0.75 and a 95% confidence interval of [165.01, 167.95]. The official U.S. yield reported was 179.3 bu/acre, which falls well outside our confidence interval, confirming that our models are not reliable for yield prediction.

How to improve?

- Define a specific time window for each state 
- The weather could have lasting impacts on soils, therefore we may begin the time window a little bit earlier in the season
- Restric on fewer years due to technology changes
- Build neural networks models, however the history of weather data available is not big enough to get satisfying results



- Regarding features, we could have added metrics such as crop condition which is strongly correlated with final yields. Another idea could be to look at traffic cameras and see the what are the field looking like from the ground